# **LANDUSE DATASET**

# **1. 래스터(TIF) 데이터 집계: Zonal Statistics (구역 통계)**
- .tif 데이터를 L2(시/군/구) 행정구역(Shapefile) 단위로 변환합니다.
- 구역통계 방법 : 폴리곤 내에 포함되는 래스터 픽셀 값들의 평균 계산
- shapefile은 GROW-Africa database의 shapefile 이용


In [ ]:
#google drive의 COSE362-term-project/dataset 폴더와 연결
from google.colab import drive

drive.mount('/content/drive')

!ls /content/drive/MyDrive/COSE362-term-project/dataset

import sys

sys.path.append('/content/drive/MyDrive/COSE362-term-project/dataset')

import os

os.chdir("/content/drive/MyDrive/COSE362-term-project/dataset")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
baseline  crop	landuse  Shapefiles


In [ ]:
!ls

baseline  crop	landuse  Shapefiles


In [ ]:
!pip install rasterstats

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd
import glob
import numpy as np

# 1. 지역구 지도(Shapefile)를 불러옵니다 (GROW-Africa)
regions = gpd.read_file("Shapefiles/GADM_level1_ECG.shp")

# 2. 1km 평균 tif 파일
# africa_1km_avg_{year}
landuse_1km_raster = "landuse/africa_1km_avg_2019.tif"

# 3. 구역 통계 실행 (sum/count 대신 'mean'을 요청)
#    (이 TIF 파일 자체가 0~256 사이의 평균값이므로, 'mean'을 구하면 됨)
print("Zonal statistics 시작... ")
stats = zonal_stats(regions, landuse_1km_raster, stats="mean")

# 4. 'mean' 값만 추출
mean_values = [s['mean'] if s and s['mean'] else 0 for s in stats]

# 5. 최종 '정착률' 계산 (0.0 ~ 1.0)
#    (픽셀 값이 0~256 사이의 평균값이므로, 256으로 나누면 비율이 됨)
regions['rate_landuse'] = [val / 256 for val in mean_values]

print("모든 계산 완료!")

sorted_df = regions.sort_values(by='rate_landuse',ascending=False)
print(sorted_df[['GID_1', 'COUNTRY','NAME_1','rate_landuse']].head())

Zonal statistics 시작... 
모든 계산 완료!
        GID_1                COUNTRY        NAME_1  rate_landuse
618  UGA.16_1                 Uganda       Kampala      0.953443
298  COG.10_1  Republic of the Congo  Pointe Noire      0.948722
42    BEN.8_1                  Benin      Littoral      0.846724
179   MLI.1_1                   Mali        Bamako      0.766941
293   COG.2_1  Republic of the Congo   Brazzaville      0.737545


In [ ]:
regions.columns

Index(['fid', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'VARNAME_1', 'NL_NAME_1',
       'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'GID', 'geometry'],
      dtype='object')

'landuse/WSF2015_v2_0_6_300m_max.tif'

# **2.csv 변환하기**

In [ ]:
final_features_df = regions[[
    'GID_1',              # 1. 마스터 키 (JOIN용)
    'COUNTRY',            # 2. 참고용 (국가)
    'NAME_1',             # 3. 참고용 (지역 이름)
    'rate_landuse',  # 4. 첫 번째 피처 (X1)
]]

# 2. 깨끗한 CSV 파일로 저장
# landuse_{year}.csv
final_features_df.to_csv("landuse_2019.csv", index=False)

print("최종 피처 테이블 저장 완료!")

최종 피처 테이블 저장 완료!
